Mounting Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Install Dependencies

In [ ]:
# !pip3 install timer 
#!pip3 install tensorflow==1.14 # newer version of tensorflow may not work properly


# Restart notebook if the following error pops out.
# ValueError: Variable conv1/weights already exists, disallowed. 
# Did you mean to set reuse=True or reuse=tf.AUTO_REUSE in VarScope? Originally defined at:



In [ ]:
cd /content/drive/MyDrive/tiny-tf

/content/drive/MyDrive/tiny-tf


Importing Libraries

In [ ]:
"""Training a tiny face detection network based on WIDER Face dataset."""

import numpy as np
import os.path
from lib.tiny.train import get_training_roidb, train_net
from time import strftime, localtime
import sys,os
from lib.tiny.demo import demo_net
from lib.tiny.config import cfg, cfg_from_file
from lib.networks.factory import get_network
import scipy.io
import pprint
import time
import tensorflow as tf

/usr/local/lib/python3.7/dist-packages/Cython/Compiler/Main.py:369: FutureWarning: Cython directive 'language_level' not set, using 2 for now (Py2). This will change in a later release! File: /content/drive/MyDrive/tiny-tf/lib/roi_data_layer/minibatch.pyx
  tree = Parsing.p_module(s, pxd, full_module_name)


ModuleNotFoundError: ignored

Defining Variables

In [ ]:
gpu_id = 0
max_epochs = 10
dir_path = '/content/drive/MyDrive/tiny-tf'
cfg_file = 'cfgs/tiny_resnet101.yml'
network_name = 'Resnet101_test'
model = 'output/Resnet101_tiny/'
pretrained_model = 'data/pretrain_model/Resnet101.npy' 
network_name_train = 'Resnet101_train' 
pkl_name = 'data/pickles/wider_train.pkl'
refBox = 'data/RefBox_N25_scaled.mat'
log_dir = 'log'
dataset_path = 'data'
output_dir = 'output'
restore = 0
restore_dir = None

Downloading Tiny Face Detection pretrained models on Wider Face Dataset

In [ ]:
if os.path.exists('data/pickles') is False:
  print("Downloading Pickle Models")
  os.makedirs('data/pickles', exist_ok=True)
  !wget  https://drive.google.com/file/d/10INpEMUSRwdGJaFUOHFO0Nau00rHt4A7/view?usp=share_link -O /content/drive/MyDrive/tiny-tf/data/pickles/wider_train.pkl


if os.path.exists('data/pretrain_model') is False:
  print("Downloading Pickle Models")
  os.makedirs('data/pretrain_model', exist_ok=True)
  !wget  https://drive.google.com/file/d/1RlYtu65VaZgOv4_snaNeuVE3QbQGMJcR/view?usp=sharing -O /content/drive/MyDrive/tiny-tf/data/pretrain_model/Resnet101.npy
else:
  print("Models already Downloaded")

Training

In [ ]:
#Training script doesnot take wider face dataset set directly but takes wider_train.pkl file to get started with training...

def get_output_dir(output_dir, network_name_train):
    outdir = os.path.abspath(os.path.join(os.path.dirname(dir_path), output_dir))
    time = strftime("%m-%d-%H-%M", localtime())
    if network_name_train is not None:
        n_name = network_name_train.split('_')[0]
        outdir = os.path.join(outdir, (n_name + '_' + time))
    else:
        outdir = os.path.join(outdir, time)
    if not os.path.exists(outdir):
        os.makedirs(outdir)
    return outdir

def get_log_dir(log_dir):
    logdir = os.path.abspath(os.path.join(os.path.dirname(dir_path), log_dir))
    logdir = os.path.join(logdir, strftime("%Y-%m-%d-%H-%M-%S", localtime()))
    if not os.path.exists(log_dir):
        os.makedirs(log_dir)
    return log_dir



if cfg_file is not None:
    cfg_from_file(cfg_file)

print('Using config:')
pprint.pprint(cfg)

print('Loaded pickles `{:s}` for training'.format(os.path.basename(pkl_name)))
roidb = get_training_roidb(pkl_name)

if ((restore) and (restore_dir is not None)):
    output_dir = os.path.abspath(restore_dir)
else:
    output_dir = get_output_dir(output_dir, network_name_train)

log_dir = get_log_dir(log_dir)
print('Output will be saved to `{:s}`'.format(output_dir))
print('Logs will be saved to `{:s}`'.format(log_dir))

device_name = '/gpu:{:d}'.format(gpu_id)
print(device_name)

network = get_network(network_name_train)
print('Use network `{:s}` in training'.format(network_name_train))

refBox_file = os.path.abspath(refBox)
centers_read = scipy.io.loadmat(refBox_file)['clusters']
centers_read = centers_read.astype(np.float32)
print('Use reference box file `{:s}` in training'.format(os.path.basename(refBox_file)))
print('Load done!')

train_net(network, roidb,
          output_dir=output_dir,
          log_dir=log_dir,
          ref_Box=centers_read,
          pretrained_model=pretrained_model,
          epochs=max_epochs,
          restore=bool(int(restore)))


Using config:
{'DEMO': {'CONFIDENCE_Thresh': 0.5,
          'DRAW_SCORE_COLORBAR': True,
          'MAX_INPUT_DIM': 5000,
          'NMS_Thresh': 0.1,
          'PRUNING': True,
          'VISUALIZE': True},
 'RGB_MEANS': array([119.2996 , 110.54627, 101.83843], dtype=float32),
 'RGB_VARIANCE': array([[ 0.7421559 , -1.3868568 ,  0.69416434],
       [ 2.6491976 ,  0.08836862, -2.6558025 ],
       [ 7.305443  ,  7.6848936 ,  7.54298   ]], dtype=float32),
 'TEST': {'CONFIDENCE_Thresh': 0.03,
          'GEN_PR_CURVE_TXT': True,
          'NMS_Thresh': 0.3,
          'PRUNING': True,
          'RATIO_RANGE': [0.25, 0.5, 1, 2]},
 'TRAIN': {'BATCH_SIZE': 8,
           'CONFIDENCE_Thresh': 0.5,
           'CROP_SIZE': 500,
           'DISPLAY': 1,
           'FLIPPED': True,
           'GAMMA': 0.1,
           'LEARNING_RATE': 0.0001,
           'LOG_IMAGE_ITERS': 805,
           'MOMENTUM': 0.9,
           'NEG_IOU_Thresh': 0.3,
           'NMS_Thresh': 0.1,
           'NORMALIZE': True,
    

Loaded roidb done!
Output will be saved to `/content/drive/MyDrive/output/Resnet101_11-10-11-39`
Logs will be saved to `log`
/gpu:0
Tensor("data:0", shape=(?, ?, ?, 3), dtype=float32)


The TensorFlow contrib module will not be included in TensorFlow 2.0.
For more information, please see:
  * https://github.com/tensorflow/community/blob/master/rfcs/20180907-contrib-sunset.md
  * https://github.com/tensorflow/addons
  * https://github.com/tensorflow/io (for I/O related ops)
If you depend on functionality not listed there, please file an issue.




Tensor("pool1:0", shape=(?, ?, ?, 64), dtype=float32)
Tensor("bn2a_branch1/Identity:0", shape=(?, ?, ?, 256), dtype=float32)
Tensor("bn2a_branch2c/Identity:0", shape=(?, ?, ?, 256), dtype=float32)
Tensor("res2a_relu:0", shape=(?, ?, ?, 256), dtype=float32)
Tensor("bn2b_branch2c/Identity:0", shape=(?, ?, ?, 256), dtype=float32)
Tensor("res2b_relu:0", shape=(?, ?, ?, 256), dtype=float32)
Tensor("bn2c_branch2c/Identity:0", shape=(?, ?, ?, 256), dtype=float32)
Tensor("res2c_relu:0", shape=(?, ?, ?, 256), dtype=float32)
Tensor("bn3a_branch1/Identity:0", shape=(?, ?, ?, 512), dtype=float32)
Tensor("bn3a_branch2c/Identity:0", shape=(?, ?, ?, 512), dtype=float32)
Tensor("res3a_relu:0", shape=(?, ?, ?, 512), dtype=float32)
Tensor("bn3b1_branch2c/Identity:0", shape=(?, ?, ?, 512), dtype=float32)
Tensor("res3b1_relu:0", shape=(?, ?, ?, 512), dtype=float32)
Tensor("bn3b2_branch2c/Identity:0", shape=(?, ?, ?, 512), dtype=float32)
Tensor("res3b2_relu:0", shape=(?, ?, ?, 512), dtype=float32)
Tensor("

Tensor("res4b21_relu:0", shape=(?, ?, ?, 1024), dtype=float32)
Tensor("bn4b22_branch2c/Identity:0", shape=(?, ?, ?, 1024), dtype=float32)
Tensor("res4b22_relu:0", shape=(?, ?, ?, 1024), dtype=float32)
Tensor("res3b3_relu:0", shape=(?, ?, ?, 512), dtype=float32)
Tensor("score_res4/BiasAdd:0", shape=(?, ?, ?, 125), dtype=float32)
Tensor("score4f/ResizeBilinear:0", shape=(?, ?, ?, 125), dtype=float32)
Tensor("score_res3/BiasAdd:0", shape=(?, ?, ?, 125), dtype=float32)
Tensor("score_map:0", shape=(?, ?, ?, 125), dtype=float32)
Tensor("score_map:0", shape=(?, ?, ?, 125), dtype=float32)
Tensor("score_cls/Slice:0", shape=(?, ?, ?, 25), dtype=float32)
Use network `Resnet101_train` in training
Use reference box file `RefBox_N25_scaled.mat` in training
Load done!



Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where





Solving...


/usr/local/lib/python3.7/dist-packages/tensorflow/python/ops/gradients_util.py:93: UserWarning: Converting sparse IndexedSlices to a dense Tensor of unknown shape. This may consume a large amount of memory.
  "Converting sparse IndexedSlices to a dense Tensor of unknown shape. "
/usr/local/lib/python3.7/dist-packages/tensorflow/python/ops/gradients_util.py:93: UserWarning: Converting sparse IndexedSlices to a dense Tensor of unknown shape. This may consume a large amount of memory.
  "Converting sparse IndexedSlices to a dense Tensor of unknown shape. "


Loading pretrained model weights from data/pretrain_model/Resnet101.npy
assign pretrain model weights to res2b_branch2a
assign pretrain model weights to res2b_branch2c
assign pretrain model weights to res2b_branch2b
assign pretrain model moving_mean to bn4b18_branch2c
assign pretrain model beta to bn4b18_branch2c
assign pretrain model moving_variance to bn4b18_branch2c
assign pretrain model gamma to bn4b18_branch2c
assign pretrain model weights to res4b12_branch2a
assign pretrain model weights to res3b1_branch2a
assign pretrain model weights to res3b1_branch2b
assign pretrain model weights to res3b1_branch2c
assign pretrain model weights to res4b9_branch2a
assign pretrain model weights to res4b16_branch2c
assign pretrain model weights to res4b9_branch2c
assign pretrain model weights to res4b9_branch2b
assign pretrain model weights to res2a_branch2a
assign pretrain model weights to res2a_branch2b
assign pretrain model weights to res2a_branch2c
assign pretrain model moving_mean to bn2a_b

Inference

In [ ]:
if cfg_file is not None:
    cfg_from_file(cfg_file)

weights_filename = os.path.splitext(os.path.basename(model))[0]
checkpoint_dir = os.path.abspath(model)

refBox_file = os.path.abspath(refBox)
centers_ref = scipy.io.loadmat(refBox_file)['clusters']
print('Use reference box file `{:s}`.'.format(os.path.basename(refBox_file)))
print('Load done!')

device_name = '/gpu:{:d}'.format(gpu_id)
print(device_name)

network = get_network(network_name)
print('Use network `{:s}` for testing'.format(network_name))

cfg.GPU_ID = gpu_id


# Load a trained ckpt file, NOT compatible with .npy file (The following paragraph)
try:
    ckpt = tf.train.get_checkpoint_state(checkpoint_dir)
    saver = tf.train.Saver()
    sess = tf.Session(config=tf.ConfigProto(allow_soft_placement=True))
    if ckpt and ckpt.model_checkpoint_path:
        ckpt_name = os.path.basename(ckpt.model_checkpoint_path)
        saver.restore(sess, os.path.join(checkpoint_dir, ckpt_name))
        print('Success to load checkpoint from {}'.format(ckpt_name))
    else:
        print('Failed to find a checkpoint!')
except:
    print('Error reading checkpoint file!')
    sys.exit(1)
demo_net(sess, network, centers_ref, weights_filename, dir_path)